# VideoSense — Multimodal Video Intelligence (DATA 255 Extra Credit HW3)

**Notebook plan** (built one step at a time):

| Step | Content |
|---|---|
| **1** | **Environment setup, data collection, manifest & integrity checks** ← you are here |
| 2 | Exploratory data analysis (frame counts, FPS, resolution, durations, class balance) |
| 3 | Preprocessing: frame sampling, normalization, augmentation, caching |
| 4 | Optical flow computation & visualization |
| 5 | Baseline: C3D from scratch + evaluation |
| 6 | Pretrained R(2+1)D fine-tuning & comparison |
| 7+ | Siamese network, two-stream fusion, interpretation |

## Step 1 — Environment setup & data collection

**Goal of this step:** end up with (a) the raw UCF101 videos on fast local disk, (b) a safe copy on Google Drive so we never download 6.5 GB twice, and (c) a single **manifest table** describing every video: its path, class, label, split, group and clip id.

> 💡 **Runtime choice:** this step (and EDA, preprocessing, and optical flow) needs **no GPU**. Use a **CPU runtime** for now (`Runtime → Change runtime type → CPU`) so you save your limited free GPU quota for training. Every cell here is *idempotent*: re-running it after a disconnect skips any work that's already done.

### 1.1 Check the runtime

In [ ]:
import sys, shutil, platform
print("Python :", sys.version.split()[0])
print("OS     :", platform.platform())

total, used, free = shutil.disk_usage("/content")
print(f"Local disk free: {free/1e9:.1f} GB  (we need ~15 GB: 6.5 GB rar + ~6.5 GB extracted)")

!nvidia-smi -L 2>/dev/null || echo "No GPU attached (fine for Step 1)"

### 1.2 Mount Google Drive and define paths

**Design decision: Drive for persistence, local disk for speed.**
Colab's `/content` disk is fast but is wiped when the session ends. Google Drive persists, but it is mounted over a network filesystem, so reading thousands of small video files from it during training is *very* slow.

The pattern we'll use throughout:
- **Drive** stores *a few large files*: the `.rar` archive, the manifests, and later the preprocessed caches and model checkpoints.
- **Local disk** holds the *extracted* working copy, which we rebuild at the start of each session (copy rar from Drive → extract, about 3–5 min).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_ROOT = '/content/drive/MyDrive/videosense'      # persistent
LOCAL_ROOT = '/content/videosense'                    # fast, ephemeral

DRIVE_RAW   = f'{DRIVE_ROOT}/raw'          # rar + zip archives
DRIVE_META  = f'{DRIVE_ROOT}/metadata'     # manifests, class maps
LOCAL_DATA  = f'{LOCAL_ROOT}/data'
VIDEO_DIR   = f'{LOCAL_DATA}/UCF101'       # extracted videos: UCF101/<Class>/<video>.avi
SPLITS_DIR  = f'{LOCAL_DATA}/splits'

for d in [DRIVE_RAW, DRIVE_META, LOCAL_DATA, SPLITS_DIR]:
    os.makedirs(d, exist_ok=True)
print("Paths ready.")

### 1.3 Download the archives (once, ever)

The `ensure_file` helper checks for a valid file in this order: **local → Drive → download**. A size threshold catches partial downloads, and `wget -c` resumes an interrupted download instead of restarting it.

`--no-check-certificate` is included because the CRCV server has a history of certificate problems. If the server is down or very slow, the same dataset is mirrored on Kaggle (search "UCF101"). Download it there and drop `UCF101.rar` into `MyDrive/videosense/raw/`; the cell below will pick it up automatically.

In [ ]:
import subprocess

RAR_URL    = 'https://www.crcv.ucf.edu/data/UCF101/UCF101.rar'
SPLITS_URL = 'https://www.crcv.ucf.edu/data/UCF101/UCF101TrainTestSplits-RecognitionTask.zip'

LOCAL_RAR    = f'{LOCAL_DATA}/UCF101.rar'
LOCAL_SPLITS = f'{LOCAL_DATA}/UCF101TrainTestSplits.zip'

def ensure_file(url, local_path, drive_path, min_bytes):
    """Make sure a valid copy exists locally; use Drive as cache; download as last resort."""
    ok = lambda p: os.path.exists(p) and os.path.getsize(p) >= min_bytes
    if ok(local_path):
        print(f'[local]    {os.path.basename(local_path)} already present')
    elif ok(drive_path):
        print(f'[drive]    copying {os.path.basename(drive_path)} to local disk...')
        shutil.copy(drive_path, local_path)
    else:
        print(f'[download] {url}')
        subprocess.run(['wget', '-c', '--no-check-certificate', '--progress=dot:giga',
                        '-O', local_path, url], check=True)
        print(f'[drive]    backing up to {drive_path}')
        shutil.copy(local_path, drive_path)
    assert ok(local_path), f'{local_path} missing or truncated!'
    print(f'           size = {os.path.getsize(local_path)/1e9:.2f} GB')

ensure_file(SPLITS_URL, LOCAL_SPLITS, f'{DRIVE_RAW}/UCF101TrainTestSplits.zip', min_bytes=50_000)
ensure_file(RAR_URL,    LOCAL_RAR,    f'{DRIVE_RAW}/UCF101.rar',               min_bytes=6_000_000_000)

### 1.4 Extract the archives

The `.rar` contains a top-level folder named `UCF-101/`. We rename it to `UCF101/` to match the assignment's directory layout. Extraction is skipped if all 13,320 videos are already there.

In [ ]:
import glob, zipfile

EXPECTED_VIDEOS, EXPECTED_CLASSES = 13_320, 101

def count_videos():
    return len(glob.glob(f'{VIDEO_DIR}/*/*.avi'))

if count_videos() < EXPECTED_VIDEOS:
    if shutil.which('unrar') is None:
        subprocess.run('apt-get -qq install -y unrar > /dev/null', shell=True, check=True)
    print('Extracting videos (~2-4 min)...')
    subprocess.run(['unrar', 'x', '-o+', '-inul', LOCAL_RAR, LOCAL_DATA + '/'], check=True)
    if os.path.isdir(f'{LOCAL_DATA}/UCF-101'):
        if os.path.isdir(VIDEO_DIR):
            shutil.rmtree(VIDEO_DIR)
        os.rename(f'{LOCAL_DATA}/UCF-101', VIDEO_DIR)
else:
    print('Videos already extracted.')

with zipfile.ZipFile(LOCAL_SPLITS) as z:
    z.extractall(SPLITS_DIR)
SPLIT_FILES = f'{SPLITS_DIR}/ucfTrainTestlist'   # the zip extracts into this sub-folder
print('Split files:', sorted(os.listdir(SPLIT_FILES)))

### 1.5 Integrity check — never trust a download blindly

In [ ]:
class_dirs = sorted(d for d in os.listdir(VIDEO_DIR) if os.path.isdir(f'{VIDEO_DIR}/{d}'))
n_videos = count_videos()
print(f'Classes found: {len(class_dirs)} (expected {EXPECTED_CLASSES})')
print(f'Videos found : {n_videos} (expected {EXPECTED_VIDEOS})')
assert len(class_dirs) == EXPECTED_CLASSES and n_videos == EXPECTED_VIDEOS, 'Incomplete extraction!'
print('✅ Dataset complete')

### 1.6 Build the manifest: one row per video

This is the most important cell of Step 1, and it's where a few decisions matter.

**Decision 1: the split files are not consistent.** `trainlist01.txt` lines look like `ApplyEyeMakeup/v_ApplyEyeMakeup_g08_c01.avi 1`, but `testlist01.txt` lines contain **only the path, no label**. The assignment's `_load_split` keeps only lines with ≥ 2 fields, so it would silently produce an **empty test set**. We derive every label from the **class folder name** via `classInd.txt` instead. That works the same for both splits, and for train we cross-check it against the given label.

**Decision 2: parse group and clip ids from the filename** (`v_<Class>_g<group>_c<clip>.avi`). A *group* is a set of clips cut from the **same long source video**: same person, same background, same camera. Clips within a group are near-duplicates.

**Decision 3: path matching is case-insensitive.** Folder names and filenames in UCF101 don't always agree on capitalization across mirrors and operating systems. We resolve each listed path against what actually exists on disk and report anything that can't be found.

**Decision 4: store labels 0-indexed** (PyTorch's `CrossEntropyLoss` expects `0..C-1`).

In [ ]:
import re
import pandas as pd

# --- class index: "1 ApplyEyeMakeup" -> 0-indexed label
class_ind = {}
with open(f'{SPLIT_FILES}/classInd.txt') as f:
    for line in f:
        if line.strip():
            idx, name = line.split()
            class_ind[name.lower()] = (int(idx) - 1, name)

# --- every video actually on disk, keyed case-insensitively
on_disk = {os.path.relpath(p, VIDEO_DIR).lower(): os.path.relpath(p, VIDEO_DIR)
           for p in glob.glob(f'{VIDEO_DIR}/*/*.avi')}

FNAME_RE = re.compile(r'v_(?P<cls>.+)_g(?P<group>\d+)_c(?P<clip>\d+)\.avi$', re.IGNORECASE)

def parse_list(list_file, split_name):
    rows, missing = [], []
    with open(list_file) as f:
        for line in f:
            parts = line.strip().split()
            if not parts:
                continue
            listed_path = parts[0]
            given_label = int(parts[1]) - 1 if len(parts) > 1 else None   # testlist has no label

            actual = on_disk.get(listed_path.lower())
            if actual is None:
                missing.append(listed_path); continue

            folder = actual.split('/')[0]
            label, class_name = class_ind[folder.lower()]
            if given_label is not None:
                assert given_label == label, f'Label mismatch for {listed_path}'

            m = FNAME_RE.search(os.path.basename(actual))
            rows.append(dict(rel_path=actual, class_name=class_name, label=label,
                             split=split_name, group=int(m['group']), clip=int(m['clip'])))
    if missing:
        print(f'⚠️  {len(missing)} listed files not found in {split_name}, e.g. {missing[:3]}')
    return rows

manifest = pd.DataFrame(parse_list(f'{SPLIT_FILES}/trainlist01.txt', 'train') +
                        parse_list(f'{SPLIT_FILES}/testlist01.txt',  'test'))
print(manifest.shape)
manifest.head()

### 1.7 Verify the split: completeness and **no leakage**

**Why the official split is group-based, and why we must not reshuffle it.** If clips from the same group landed in both train and test, the model could score well by recognizing *the person or the room* rather than *the action*. Your test accuracy would be inflated and would collapse on new footage. Split 1 puts groups 1–7 in test and 8–25 in train.

This is also why we will later carve a **validation set out of train by group**, never by random clip.

In [ ]:
# 1) every video assigned exactly once
assert manifest.rel_path.is_unique, 'A video appears twice!'
assert len(manifest) == EXPECTED_VIDEOS, f'{EXPECTED_VIDEOS - len(manifest)} videos unassigned'

# 2) no group shared between train and test within any class
leaks = []
for cls, df in manifest.groupby('class_name'):
    shared = set(df[df.split == 'train'].group) & set(df[df.split == 'test'].group)
    if shared:
        leaks.append((cls, shared))
assert not leaks, f'Group leakage: {leaks[:5]}'

print('✅ All videos assigned once, no group leakage')
print(manifest.split.value_counts().to_string())
print('\nTest groups :', sorted(manifest[manifest.split == 'test'].group.unique()))
print('Train groups:', sorted(manifest[manifest.split == 'train'].group.unique()))

### 1.8 A first look at class balance

The full EDA is Step 2. For now we just want to know whether any class is severely under-represented, because that would change our metric choice (plain accuracy vs. macro-F1) and our sampling strategy.

In [ ]:
counts = manifest.pivot_table(index='class_name', columns='split', values='rel_path',
                              aggfunc='count').fillna(0).astype(int)
counts['total'] = counts.sum(axis=1)
print(counts.total.describe().round(1).to_string())
print('\nSmallest classes:\n', counts.sort_values('total').head(5).to_string())
print('\nLargest classes:\n',  counts.sort_values('total').tail(5).to_string())
print(f'\nImbalance ratio (max/min): {counts.total.max()/counts.total.min():.2f}')

### 1.9 Create the 10-class mini subset

We develop and debug on 10 classes (minutes per run) and move to 101 classes only for final runs.

**Decision: remap labels to `0..9`.** The original labels are scattered across 0–100. With a 10-output classifier, a label like 37 crashes `CrossEntropyLoss` (`IndexError` on CUDA, or a cryptic device-side assert). We remap using the **sorted** class names so the mapping is deterministic across sessions.

We do **not** copy video files (the assignment's `shutil.copytree` approach). The mini subset is just a filtered manifest pointing at the same files: no duplicate disk usage, and one source of truth.

In [ ]:
import json

MINI_CLASSES = sorted(['WalkingWithDog', 'Basketball', 'PushUps', 'Fencing', 'HandstandWalking',
                       'Surfing', 'BabyCrawling', 'Archery', 'PullUps', 'YoYo'])
missing_cls = set(MINI_CLASSES) - set(manifest.class_name)
assert not missing_cls, f'Unknown classes: {missing_cls}'

mini = manifest[manifest.class_name.isin(MINI_CLASSES)].copy()
mini_map = {c: i for i, c in enumerate(MINI_CLASSES)}
mini['orig_label'] = mini.label
mini['label'] = mini.class_name.map(mini_map)

print(mini.groupby(['class_name', 'split']).size().unstack().to_string())
print(f'\nMini subset: {len(mini)} videos '
      f'({(mini.split=="train").sum()} train / {(mini.split=="test").sum()} test)')

### 1.10 Save everything to Drive

Later steps load these CSVs instead of re-parsing the split files. If you start a new session, you only need cells 1.1–1.4 (to restore the local videos) plus loading these files.

In [ ]:
manifest.to_csv(f'{DRIVE_META}/manifest_full.csv', index=False)
mini.to_csv(f'{DRIVE_META}/manifest_mini.csv', index=False)

full_map = {name: idx for idx, name in sorted(class_ind.values())}
with open(f'{DRIVE_META}/class_maps.json', 'w') as f:
    json.dump({'full': full_map, 'mini': mini_map}, f, indent=2)

print('Saved to', DRIVE_META, ':', os.listdir(DRIVE_META))

---
### ✅ Step 1 checklist
- [ ] 101 classes / 13,320 videos verified
- [ ] Manifest built with labels for **both** train and test
- [ ] No group leakage between train and test
- [ ] Mini subset created with labels remapped to 0–9
- [ ] Manifests saved to Drive

**Next — Step 2 (EDA):** we'll read video metadata (frame count, FPS, resolution, duration) for every clip. The numbers we find there are what justify choices like *T = 16 frames* and *112×112* in the assignment.

## Step 2 — Exploratory Data Analysis (EDA)

**Goal:** before choosing any preprocessing parameter (frames per clip, resolution, how to sample, what to cache), measure the data. Every choice we make in Step 3 should trace back to a number produced here.

**Questions this step answers**
1. What do the videos look like technically (resolution, FPS, codec, length)?
2. Can we trust the frame count stored in the file header, and is seeking to a frame accurate and fast?
3. What does "sample 16 frames" actually mean in seconds for these clips?
4. How balanced are the classes, and how many truly independent samples do we have?
5. Which classes are motion-heavy, and where is there camera motion? (This sets up optical flow.)
6. How much disk will our preprocessed cache need?

> Still a **CPU runtime**. In a new session, run cells 1.1–1.4 first to restore the videos to local disk.

### 2.0 Load the manifests from Step 1

In [ ]:
import os, json, glob, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cv2
from concurrent.futures import ThreadPoolExecutor
from tqdm.auto import tqdm

DRIVE_FIG = f'{DRIVE_ROOT}/figures'          # figures for the report (deliverable 11.4)
os.makedirs(DRIVE_FIG, exist_ok=True)

def save_fig(name):
    plt.savefig(f'{DRIVE_FIG}/{name}.png', dpi=150, bbox_inches='tight')

manifest = pd.read_csv(f'{DRIVE_META}/manifest_full.csv')
mini     = pd.read_csv(f'{DRIVE_META}/manifest_mini.csv')
assert len(glob.glob(f'{VIDEO_DIR}/*/*.avi')) == EXPECTED_VIDEOS, 'Videos missing: run cells 1.1-1.4 first'
print('Full manifest:', manifest.shape, '| Mini manifest:', mini.shape)

### 2.1 Read the header metadata of every video

Opening a video and reading its header is cheap (no decoding), so we can do all 13,320 in a few minutes. Threads work well here because OpenCV releases Python's GIL during I/O. The result is cached to Drive, so it runs only once.

In [ ]:
META_CSV = f'{DRIVE_META}/video_meta.csv'

def read_meta(rel):
    cap = cv2.VideoCapture(f'{VIDEO_DIR}/{rel}')
    if not cap.isOpened():
        return dict(rel_path=rel, readable=False)
    fourcc = int(cap.get(cv2.CAP_PROP_FOURCC))
    meta = dict(rel_path=rel, readable=True,
                n_frames=int(cap.get(cv2.CAP_PROP_FRAME_COUNT)),
                fps=round(cap.get(cv2.CAP_PROP_FPS), 2),
                width=int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)),
                height=int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT)),
                codec=''.join(chr((fourcc >> 8 * i) & 0xFF) for i in range(4)))
    cap.release()
    return meta

if os.path.exists(META_CSV):
    vmeta = pd.read_csv(META_CSV)
    print('Loaded cached metadata')
else:
    with ThreadPoolExecutor(max_workers=8) as ex:
        rows = list(tqdm(ex.map(read_meta, manifest.rel_path), total=len(manifest), desc='Reading headers'))
    vmeta = pd.DataFrame(rows)
    vmeta.to_csv(META_CSV, index=False)

df = manifest.merge(vmeta, on='rel_path')
print(f'Unreadable videos: {(~df.readable.astype(bool)).sum()}')
df = df[df.readable.astype(bool)].copy()
df['duration_s'] = df.n_frames / df.fps
df[['n_frames', 'fps', 'width', 'height', 'duration_s']].describe().round(2)

### 2.2 Resolution, frame rate and codec

In [ ]:
print('Codec:\n', df.codec.value_counts().to_string(), '\n')
print('FPS:\n', df.fps.value_counts().head(10).to_string(), '\n')
res = (df.width.astype(str) + 'x' + df.height.astype(str)).value_counts()
print('Resolution:\n', res.head(10).to_string(), '\n')
print('Aspect ratio (w/h):', (df.width / df.height).round(3).value_counts().to_dict())

**What to look for, and why it matters**

- **Aspect ratio.** UCF101 is essentially all 320×240, which is **4:3**. The assignment's loader does `cv2.resize(frame, (112, 112))`, which squashes every frame horizontally by 25%. People look thinner and motions look compressed sideways. The C3D paper avoided this by resizing to **128×171** (keeps 4:3) and then **cropping 112×112**. A crop also gives us free augmentation: random crop for training, center crop for testing. We'll adopt this in Step 3.
- **FPS.** If almost everything is 25 FPS, "frame index" and "time" are interchangeable. If a few clips are ~30 FPS, a fixed frame gap means slightly different real time gaps. That's minor here, but it's the reason we measure in seconds below.
- **Codec.** UCF101 uses XviD (MPEG-4). That matters for 2.4, because seeking in MPEG-4 video depends on keyframes.

### 2.3 How long are the clips?

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].hist(df.duration_s, bins=80, color='steelblue')
axes[0].axvline(df.duration_s.median(), color='red', ls='--',
                label=f'median = {df.duration_s.median():.1f} s')
axes[0].set(xlabel='Duration (s)', ylabel='# videos', title='Clip duration'); axes[0].legend()
axes[1].hist(df.n_frames, bins=80, color='darkorange', log=True)
axes[1].set(xlabel='# frames', ylabel='# videos (log scale)', title='Frames per clip')
plt.tight_layout(); save_fig('eda_duration'); plt.show()

# Per-class duration, sorted by median: do some actions come in systematically longer clips?
order = df.groupby('class_name').duration_s.median().sort_values().index
plt.figure(figsize=(22, 5))
plt.boxplot([df[df.class_name == c].duration_s for c in order], showfliers=False)
plt.xticks(range(1, len(order) + 1), order, rotation=90, fontsize=7)
plt.ylabel('Duration (s)'); plt.title('Clip duration per class (sorted by median, outliers hidden)')
plt.tight_layout(); save_fig('eda_duration_per_class'); plt.show()

print(f'Shortest clip: {df.duration_s.min():.2f}s ({df.n_frames.min()} frames) | '
      f'Longest: {df.duration_s.max():.1f}s ({df.n_frames.max()} frames)')

**Interpretation.** Clip length varies by more than an order of magnitude, and it varies *systematically by class*. That matters because if we sampled frames at a fixed rate, clip length itself would leak class information. Sampling a **fixed number** of frames (T) removes that shortcut, but it means T frames cover very different amounts of time in a short clip versus a long one. Section 2.5 quantifies this.

### 2.4 Can we trust the header, and is seeking accurate?

The assignment's loader reads frames like this: get the frame count from the header, compute 16 indices, then `cap.set(CAP_PROP_POS_FRAMES, idx)` to jump to each one. That relies on two assumptions:

1. **The header frame count is correct.** If it overstates the length, the last indices fail to read and the loader silently inserts **all-black frames**.
2. **Seeking lands on the exact frame.** MPEG-4 stores full images only at *keyframes*. Every other frame is stored as a difference from earlier frames, so jumping to frame 137 requires decoding forward from the previous keyframe. Some backends do this imprecisely, and it is always slower than reading straight through.

We test both on a random sample instead of assuming.

In [ ]:
SAMPLE_N = 40
sample = df.sample(SAMPLE_N, random_state=0).copy()

def decode_count(rel):
    """True frame count, obtained by decoding the whole video."""
    cap = cv2.VideoCapture(f'{VIDEO_DIR}/{rel}'); n = 0
    while cap.grab():
        n += 1
    cap.release()
    return n

sample['decoded'] = [decode_count(r) for r in tqdm(sample.rel_path, desc='Full decode')]
sample['header_minus_decoded'] = sample.n_frames - sample.decoded
print('Header count minus decoded count (0 = header is exact):')
print(sample.header_minus_decoded.value_counts().sort_index().to_string())

In [ ]:
def seek_check(rel, n_frames, n_probe=6):
    """Compare frames obtained by seeking vs by sequential reading at the same indices."""
    path = f'{VIDEO_DIR}/{rel}'
    probes = sorted(set(np.linspace(0, n_frames - 1, n_probe, dtype=int).tolist()))

    cap, seq, i = cv2.VideoCapture(path), {}, 0          # ground truth: read straight through
    while True:
        ok, f = cap.read()
        if not ok:
            break
        if i in probes:
            seq[i] = f
        i += 1
    cap.release()

    cap, errs = cv2.VideoCapture(path), []               # same frames via seeking
    for p in probes:
        cap.set(cv2.CAP_PROP_POS_FRAMES, p)
        ok, f = cap.read()
        if ok and p in seq:
            errs.append(float(np.abs(f.astype(np.int16) - seq[p].astype(np.int16)).mean()))
        else:
            errs.append(np.inf)                          # frame unreachable
    cap.release()
    return errs

errs = np.concatenate([seek_check(r, n) for r, n in
                       tqdm(zip(sample.rel_path, sample.n_frames), total=SAMPLE_N, desc='Seek test')])
print(f'Probed frames        : {len(errs)}')
print(f'Exact match (err<1)  : {(errs < 1).mean():.1%}')
print(f'Wrong frame (err>=1) : {((errs >= 1) & np.isfinite(errs)).mean():.1%}')
print(f'Unreachable          : {np.isinf(errs).mean():.1%}')

In [ ]:
# Speed: seek-based sampling (the assignment's approach) vs one sequential pass
T = 16

def sample_by_seek(rel, n_frames):
    cap = cv2.VideoCapture(f'{VIDEO_DIR}/{rel}')
    for idx in np.linspace(0, n_frames - 1, T, dtype=int):
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(idx)); cap.read()
    cap.release()

def sample_sequential(rel, n_frames):
    keep = set(np.linspace(0, n_frames - 1, T, dtype=int).tolist())
    cap, i = cv2.VideoCapture(f'{VIDEO_DIR}/{rel}'), 0
    while cap.grab():                 # grab() advances without the cost of converting the image
        if i in keep:
            cap.retrieve()
        i += 1
    cap.release()

timing = {}
for name, fn in [('seek', sample_by_seek), ('sequential', sample_sequential)]:
    t0 = time.perf_counter()
    for r, n in zip(sample.rel_path, sample.n_frames):
        fn(r, n)
    timing[name] = (time.perf_counter() - t0) / SAMPLE_N

n_train = (manifest.split == 'train').sum()
for name, sec in timing.items():
    print(f'{name:>10}: {sec*1000:6.1f} ms/clip -> ~{sec*n_train/60:5.1f} min of pure decoding per epoch (full train set)')

**Interpretation.** Whatever the exact numbers on your runtime, compare the per-epoch decoding time with the number of epochs the assignment asks for. 30 epochs of C3D, plus label-smoothing sweeps, plus R(2+1)D, plus Siamese and fusion runs, all re-decode the same videos. **Decoding once and caching the sampled frames as arrays** turns that per-epoch cost into a fast array load, so the GPU stops waiting on the CPU. If the seek test found any wrong or unreachable frames, that's a second, independent reason to decode sequentially. This is the core design of Step 3.

### 2.5 What does "16 frames" mean in seconds?

There are two common ways to pick T frames:

- **Uniform over the whole clip** (the assignment's approach): covers the entire action, but in a long clip neighbouring sampled frames can be more than a second apart.
- **T consecutive frames** (what the C3D paper trained on): smooth motion, with neighbouring frames 40 ms apart, but only covers a small window of a long clip.

The choice matters because C3D's **3×3×3 kernels look at 3 neighbouring frames at a time**. They're designed to detect *local motion*. If neighbouring frames are 1–2 s apart, the "motion" the kernel sees is a jump between unrelated poses.

In [ ]:
T = 16
df['uniform_gap_s']   = (df.n_frames - 1) / (T - 1) / df.fps     # time between sampled frames
df['consec_coverage'] = np.minimum(T / df.n_frames, 1.0)          # fraction of clip seen

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].hist(df.uniform_gap_s, bins=80, color='seagreen')
axes[0].axvline(1 / 25, color='red', ls='--', label='consecutive frames at 25 fps (0.04 s)')
axes[0].set(xlabel='Seconds between sampled frames', ylabel='# videos',
            title=f'Uniform sampling, T={T}: temporal gap'); axes[0].legend()
axes[1].hist(df.consec_coverage * 100, bins=80, color='slateblue')
axes[1].set(xlabel='% of clip covered', ylabel='# videos',
            title=f'{T} consecutive frames: coverage of the clip')
plt.tight_layout(); save_fig('eda_sampling_tradeoff'); plt.show()

q = df.uniform_gap_s.quantile([.1, .5, .9]).round(3)
print(f'Uniform sampling gap (s): p10={q[.1]}, median={q[.5]}, p90={q[.9]}')
print(f'Median coverage with {T} consecutive frames: {df.consec_coverage.median():.1%}')
for t in (16, 32, 64):
    print(f'Clips shorter than {t:>2} frames: {(df.n_frames < t).sum()}')

**Interpretation, and the decision it leads to.** Neither extreme is ideal: uniform sampling gives full coverage but jerky motion, while consecutive frames give smooth motion but miss most of the action. The usual middle ground is **segment-based sampling**: split the clip into T equal segments and take one frame from each. During training, pick a *random* frame inside each segment (temporal augmentation, so the model sees slightly different timings each epoch). During testing, take the *center* of each segment (deterministic). This keeps full coverage and adds variety for free.

We'll return to this trade-off in **Reflection Q1** ("Why a fixed number of frames? What might we lose?"). These plots are your evidence for that answer.

### 2.6 Class balance and group structure

In [ ]:
counts = df.pivot_table(index='class_name', columns='split', values='rel_path', aggfunc='count').fillna(0)
counts = counts.loc[counts.sum(axis=1).sort_values().index]
counts[['train', 'test']].plot.bar(stacked=True, figsize=(22, 5), width=0.85, color=['steelblue', 'orange'])
plt.ylabel('# clips'); plt.title('Clips per class (split 1)'); plt.xticks(fontsize=7)
plt.tight_layout(); save_fig('eda_class_balance'); plt.show()

tot = counts.sum(axis=1)
print(f'Clips per class: min={tot.min():.0f}, max={tot.max():.0f}, ratio={tot.max()/tot.min():.2f}')
print(f'Test fraction per class: {(counts.test / tot).min():.2f} - {(counts.test / tot).max():.2f}')

clips_per_group = df.groupby(['class_name', 'group']).size()
print(f'\nClips per (class, group): mean={clips_per_group.mean():.1f}, '
      f'min={clips_per_group.min()}, max={clips_per_group.max()}')
print(f'Independent source videos per class in train: '
      f'{df[df.split=="train"].groupby("class_name").group.nunique().median():.0f} (median)')

**Interpretation.**
- **Balance.** The classes are mildly imbalanced, not severely. Plain accuracy is still meaningful, but we'll also report **macro-F1** (required in Part 5), which weights every class equally, so a weak small class can't hide behind strong large ones.
- **Groups.** Each class has ~70–100 training clips, but they come from only **18 source videos** (groups 8–25), each cut into several near-duplicate clips. The *effective* sample size is much smaller than the clip count suggests. This explains why C3D trained from scratch overfits so easily, and why pretrained models (R(2+1)D, Q12) help so much. It also means our **validation split must hold out whole groups**, or validation accuracy will be inflated exactly the way test leakage would be.

### 2.7 Look at the data

Numbers can hide problems that are obvious to the eye: letterboxing, watermarks, near-identical scenes across classes. This cell also produces **Part 1, Exercise 4**: one clip shown as 8 uniformly sampled frames.

In [ ]:
def read_frames(rel, indices):
    """Read the requested frame indices (RGB) with a single sequential pass."""
    want, out = sorted(set(int(i) for i in indices)), {}
    cap, i = cv2.VideoCapture(f'{VIDEO_DIR}/{rel}'), 0
    while len(out) < len(want) and cap.grab():
        if i in want:
            ok, f = cap.retrieve()
            if ok:
                out[i] = cv2.cvtColor(f, cv2.COLOR_BGR2RGB)
        i += 1
    cap.release()
    return [out[i] for i in want if i in out]

# (a) One middle frame from a random training clip of each mini class
mini_meta = df[df.rel_path.isin(mini.rel_path)]
picks = mini_meta[mini_meta.split == 'train'].groupby('class_name').sample(1, random_state=1)
fig, axes = plt.subplots(2, 5, figsize=(18, 6))
for ax, (_, row) in zip(axes.flat, picks.iterrows()):
    ax.imshow(read_frames(row.rel_path, [row.n_frames // 2])[0])
    ax.set_title(row.class_name, fontsize=10); ax.axis('off')
plt.suptitle('Mini subset: one frame per class'); plt.tight_layout()
save_fig('eda_mini_class_montage'); plt.show()

# (b) Exercise 4: one clip as 8 uniformly sampled frames
row = mini_meta[mini_meta.class_name == 'PullUps'].iloc[0]
idx = np.linspace(0, row.n_frames - 1, 8, dtype=int)
fig, axes = plt.subplots(1, 8, figsize=(22, 3))
for ax, frame, i in zip(axes, read_frames(row.rel_path, idx), idx):
    ax.imshow(frame); ax.set_title(f'frame {i}\nt={i / row.fps:.2f}s', fontsize=9); ax.axis('off')
plt.suptitle(f'{row.class_name}: 8 uniformly sampled frames ({row.rel_path})')
plt.tight_layout(); save_fig('part1_ex4_frame_sampling'); plt.show()

### 2.8 A cheap motion probe (preview of optical flow)

Optical flow is expensive, so let's first get a rough idea of motion with something cheap: the **absolute pixel difference between consecutive frames**.

- **Motion energy** = average frame difference. High means a lot of change per frame.
- **Border/center ratio** = motion near the frame edges divided by motion in the center. Actors are usually near the center, so a ratio around 1 or higher hints that the *whole frame* is moving, which usually means **camera motion** (panning, a handheld camera, following a surfer).

This is a heuristic, not a measurement, but it's useful evidence for **Reflection Q2** (camera pan vs. subject motion), and it tells us which classes optical flow should help most.

In [ ]:
def motion_probe(rel, n_frames, n=24, size=(160, 120)):
    start = max(0, n_frames // 2 - n // 2)
    frames = read_frames(rel, range(start, min(start + n, n_frames)))
    gray = np.stack([cv2.resize(cv2.cvtColor(f, cv2.COLOR_RGB2GRAY), size).astype(np.float32)
                     for f in frames])
    diffs = np.abs(np.diff(gray, axis=0))                     # (n-1, H, W)
    H, W = diffs.shape[1:]
    center = np.zeros((H, W), bool)
    center[H // 4: 3 * H // 4, W // 4: 3 * W // 4] = True
    return diffs.mean(), diffs[:, ~center].mean() / (diffs[:, center].mean() + 1e-6)

probe_set = (mini_meta[mini_meta.split == 'train'].sample(frac=1, random_state=0)
             .groupby('class_name').head(8))                      # up to 8 clips per class
rows = []
for r in tqdm(probe_set.itertuples(), total=len(probe_set), desc='Motion probe'):
    energy, ratio = motion_probe(r.rel_path, r.n_frames)
    rows.append(dict(class_name=r.class_name, energy=energy, border_center=ratio))
probe = pd.DataFrame(rows)
summary = probe.groupby('class_name')[['energy', 'border_center']].mean().sort_values('energy')

fig, axes = plt.subplots(1, 2, figsize=(16, 4))
summary.energy.plot.barh(ax=axes[0], color='indianred', title='Motion energy (mean |frame difference|)')
summary.border_center.plot.barh(ax=axes[1], color='teal', title='Border / center motion ratio')
axes[1].axvline(1.0, color='k', ls='--', lw=1)
plt.tight_layout(); save_fig('eda_motion_probe'); plt.show()
summary.round(2)

**Interpretation.** Note the classes with high energy *and* a high border/center ratio (e.g. sports filmed with a moving camera). For those, raw frame differences, and also raw optical flow, are dominated by the camera rather than the person. That's exactly the problem Reflection Q2 asks about. Classes with low energy (e.g. Archery, where most of the clip is aiming) are where appearance (RGB) will likely matter more than motion. Keep these predictions; we'll check them against the Part 4 ablation (RGB-only vs. flow-only per class).

### 2.9 How much disk will the preprocessed cache need?

Caching sampled frames (decided in 2.4) has a disk cost. We compute it for the options under consideration, as uint8 (1 byte per value), before we commit.

In [ ]:
rows = []
for T_ in (16, 32):
    for (h, w) in ((112, 112), (128, 171)):
        per_clip = T_ * h * w * 3                         # bytes, uint8 RGB
        rows.append(dict(T=T_, resolution=f'{h}x{w}', MB_per_clip=per_clip / 1e6,
                         mini_GB=per_clip * len(mini) / 1e9, full_GB=per_clip * len(manifest) / 1e9))
storage = pd.DataFrame(rows).round(2)
print(f'Free local disk: {shutil.disk_usage("/content").free / 1e9:.0f} GB')
storage

**Interpretation.** Caching at **128×171** (aspect-correct, crop-ready) roughly doubles storage compared with 112×112, and T=32 doubles it again. For the mini subset any option is small. For the full dataset, we'll balance this against Colab's local disk and Drive quota in Step 3. Optical flow adds 2 channels per frame instead of 3, and we'll store it as float16.

### 2.10 Save the enriched manifest and summarize

In [ ]:
df.to_csv(f'{DRIVE_META}/manifest_with_meta.csv', index=False)

print('=== EDA summary ===')
print(f'Videos: {len(df)} | classes: {df.class_name.nunique()}')
print(f'Resolution: {res.index[0]} ({res.iloc[0] / len(df):.1%}) | FPS mode: {df.fps.mode()[0]}')
print(f'Duration: median {df.duration_s.median():.1f}s, range {df.duration_s.min():.1f}-{df.duration_s.max():.1f}s')
print(f'Header frame count exact in {(sample.header_minus_decoded == 0).mean():.0%} of sampled clips')
print(f'Seek exact in {(errs < 1).mean():.0%} of probes | seek {timing["seek"]*1000:.0f} ms vs '
      f'sequential {timing["sequential"]*1000:.0f} ms per clip')
print(f'Uniform T=16 gap: median {df.uniform_gap_s.median():.2f}s between sampled frames')
print('Figures saved to:', DRIVE_FIG)

---
### ✅ Step 2 → decisions carried into Step 3

| Finding | Decision |
|---|---|
| Frames are 320×240 (4:3) | Resize to **128×171** (aspect preserved), then **random crop 112** for train / **center crop** for test |
| Re-decoding every epoch is slow; seeking can be inexact | **Decode once, sequentially**; cache sampled frames as uint8 `.npy` |
| Uniform sampling of long clips puts > 1 s between frames | **Segment-based sampling**: random frame per segment (train), segment center (test) |
| Clip length differs by class | Keep a **fixed T** so length can't become a shortcut feature |
| ~18 source groups per class in train | **Validation split by group**; expect overfitting from scratch |
| Mild class imbalance | Report **macro-F1** alongside top-1/top-5 |
| Some classes have strong camera motion | Watch these in the flow ablation; supports Reflection Q2 |

**Next — Step 3 (Preprocessing):** build the frame cache with segment-based sampling, the group-based validation split, normalization, and augmentation transforms, then wrap it all in a PyTorch `Dataset` and check every output shape.

## Step 3 — Preprocessing: splits, frame cache, transforms, and the PyTorch Dataset

**Goal:** turn raw `.avi` files into a fast, correct stream of `(3, T, 112, 112)` tensors. Every choice here implements one of the Step 2 decisions:

| Step 2 finding | Step 3 implementation |
|---|---|
| Re-decoding / seeking every epoch is slow | **3.2–3.3** Decode each video **once**, cache frames to disk |
| Uniform sampling of long clips is jerky | **3.5** Cache K=32 frames, then **segment-sample** T=16 from them |
| Frames are 4:3 | Cache at **128×171**, **crop 112×112** (random for train, center for eval) |
| Only ~18 source groups per class | **3.1** Validation split **by group** |

> Still a **CPU runtime**. In a new session, run cells 1.1–1.4 (restore videos) and 2.0 (imports/paths) first.

### 3.0 Configuration

In [ ]:
import os, sys, random, subprocess, shutil, time
import numpy as np, pandas as pd, cv2, torch
import matplotlib.pyplot as plt
from concurrent.futures import ThreadPoolExecutor
from tqdm.auto import tqdm

SUBSET = 'mini'            # 'mini' for development; switch to 'full' for final runs
T          = 16            # frames per clip fed to the model
CACHE_K    = 32            # frames stored per clip (> T, so training can jitter frame choice)
CACHE_H, CACHE_W = 128, 171   # aspect-preserving 4:3 size (C3D paper convention)
CROP       = 112
JPEG_Q     = 90
SEED       = 42

CACHE_NAME  = f'frames_k{CACHE_K}_{CACHE_H}x{CACHE_W}_q{JPEG_Q}'   # params in the name: no stale mixing
LOCAL_CACHE = f'{LOCAL_DATA}/{CACHE_NAME}'
DRIVE_CACHE = f'{DRIVE_ROOT}/cache'
CODE_DIR    = f'{DRIVE_ROOT}/code'           # project code (deliverable 11.1 structure)
for d in [LOCAL_CACHE, DRIVE_CACHE, f'{CODE_DIR}/utils', f'{CODE_DIR}/models']:
    os.makedirs(d, exist_ok=True)
for pkg in ['utils', 'models']:
    open(f'{CODE_DIR}/{pkg}/__init__.py', 'a').close()

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
print('Subset:', SUBSET, '| cache:', CACHE_NAME, '| CPUs:', os.cpu_count())

### 3.1 Train / validation / test split — by group

**Why we need a validation set at all.** The assignment's training script evaluates on `testlist01` every epoch. If you use those numbers to pick the best epoch, label-smoothing value, or margin, you're tuning on the test set, and the final test accuracy is no longer an honest estimate. So we keep **test untouched until final evaluation** and make choices on a **validation set** carved out of train.

**Why by group.** From Step 2: clips in the same group are near-duplicates (same person, same scene). A random clip-level split would put siblings of every validation clip in train, and validation accuracy would be inflated. We hold out **whole groups 8–10** (3 of the 18 training groups, ~17%) in every class.

In [ ]:
VAL_GROUPS = {8, 9, 10}

meta_cols = pd.read_csv(f'{DRIVE_META}/manifest_with_meta.csv')[['rel_path', 'n_frames', 'fps']]
full = pd.read_csv(f'{DRIVE_META}/manifest_full.csv').merge(meta_cols, on='rel_path', how='left')
mini = pd.read_csv(f'{DRIVE_META}/manifest_mini.csv').merge(meta_cols, on='rel_path', how='left')

def add_phase(m):
    m = m.copy()
    m['phase'] = np.where(m.split == 'test', 'test',
                          np.where(m.group.isin(VAL_GROUPS), 'val', 'train'))
    for cls, d in m.groupby('class_name'):          # no group may appear in two phases
        g = {p: set(d[d.phase == p].group) for p in ('train', 'val', 'test')}
        assert not (g['train'] & g['val'] or g['train'] & g['test'] or g['val'] & g['test']), cls
    return m

full, mini = add_phase(full), add_phase(mini)
full.to_csv(f'{DRIVE_META}/splits_full.csv', index=False)
mini.to_csv(f'{DRIVE_META}/splits_mini.csv', index=False)

data = mini if SUBSET == 'mini' else full
print(data.phase.value_counts().to_string())
print('\nPer class (first 5):')
print(data.groupby(['class_name', 'phase']).size().unstack().head().to_string())

### 3.2 The cache format — design and a test on one clip

**What we store per clip:** K=32 frames sampled uniformly over the whole clip, resized to 128×171, each **JPEG-encoded**, packed into one `.npz` (all bytes + offsets).

**Why JPEG instead of raw arrays?** Step 2.9 showed raw uint8 at 128×171 with K=32 is ~2.1 MB per clip, ~28 GB for the full dataset, which is more than a free Google Drive (15 GB). JPEG at quality 90 is ~10× smaller, and decoding 16 small JPEGs takes only a few milliseconds. We trade a tiny amount of image quality for an order-of-magnitude storage saving; the PSNR check below quantifies the quality cost.

**Why K=32 > T=16?** So training can choose *different* frames each epoch (temporal augmentation, 3.5). More K means more jitter but more disk; 32 is a middle ground.

**Robustness details:**
- Frames are read in **one sequential pass** (Step 2.4: faster and exact).
- If the header overstates the length, missing indices fall back to the **nearest decoded frame** instead of a black frame (the bug in the original loader).
- Files are written to a temp name, then **atomically renamed**, so a Colab disconnect never leaves a half-written file that later looks complete.

In [ ]:
def cache_path(rel):
    return f'{LOCAL_CACHE}/{rel[:-4]}.npz'

def cache_indices(n_frames, k=CACHE_K):
    return np.linspace(0, max(int(n_frames) - 1, 0), k).round().astype(int)

def read_resized(rel, indices):
    """Sequentially decode only the requested frames (BGR, resized)."""
    want, got = set(int(i) for i in indices), {}
    cap, i = cv2.VideoCapture(f'{VIDEO_DIR}/{rel}'), 0
    while len(got) < len(want) and cap.grab():
        if i in want:
            ok, f = cap.retrieve()
            if ok:
                got[i] = cv2.resize(f, (CACHE_W, CACHE_H), interpolation=cv2.INTER_AREA)
        i += 1
    cap.release()
    return got

def build_clip_cache(rel, n_frames):
    out = cache_path(rel)
    if os.path.exists(out):
        return 'skip'
    idx = cache_indices(n_frames)
    got = read_resized(rel, idx)
    if not got:
        return 'fail'
    avail = np.array(sorted(got))
    frames = [got[avail[np.abs(avail - j).argmin()]] for j in idx]     # nearest decoded frame
    jpgs = [cv2.imencode('.jpg', f, [cv2.IMWRITE_JPEG_QUALITY, JPEG_Q])[1].ravel() for f in frames]
    offsets = np.concatenate([[0], np.cumsum([len(j) for j in jpgs])])
    os.makedirs(os.path.dirname(out), exist_ok=True)
    tmp = out + '.tmp'
    with open(tmp, 'wb') as fh:
        np.savez(fh, data=np.concatenate(jpgs), offsets=offsets, src_idx=idx)
    os.replace(tmp, out)                                               # atomic
    return 'ok' if len(got) == len(set(idx.tolist())) else 'ok_padded'

def load_cached(path, which=None):
    """Decode cached frames -> (n, H, W, 3) uint8 RGB."""
    z = np.load(path)
    data, off = z['data'], z['offsets']
    which = range(len(off) - 1) if which is None else which
    return np.stack([cv2.cvtColor(cv2.imdecode(data[off[i]:off[i + 1]], cv2.IMREAD_COLOR),
                                  cv2.COLOR_BGR2RGB) for i in which])

# --- test on one clip
row = data[data.phase == 'train'].iloc[0]
print(build_clip_cache(row.rel_path, row.n_frames), '->', cache_path(row.rel_path))
cached = load_cached(cache_path(row.rel_path))
raw = read_resized(row.rel_path, cache_indices(row.n_frames))
raw = np.stack([cv2.cvtColor(raw[i], cv2.COLOR_BGR2RGB) for i in cache_indices(row.n_frames) if i in raw])
mse = ((cached[:len(raw)].astype(np.float64) - raw) ** 2).mean()
size_kb = os.path.getsize(cache_path(row.rel_path)) / 1e3
raw_kb = CACHE_K * CACHE_H * CACHE_W * 3 / 1e3
print(f'Cached shape: {cached.shape} | file {size_kb:.0f} KB vs raw {raw_kb:.0f} KB '
      f'({raw_kb / size_kb:.1f}x smaller) | PSNR {10 * np.log10(255**2 / max(mse, 1e-9)):.1f} dB')

**Reading the PSNR:** above ~35–40 dB, compression differences are essentially invisible and far smaller than the variation from augmentation, camera noise, and the original XviD compression. If you see a much lower value, raise `JPEG_Q`.

### 3.3 Build (or restore) the cache for the whole subset

- **Restore first:** if a tar of the cache is already on Drive, extract it (seconds to minutes) instead of rebuilding.
- **Build:** threads in parallel (OpenCV releases the GIL), skipping clips already cached, so you can safely re-run after a disconnect.
- **Back up:** one `.tar` file to Drive. Thousands of small files on Drive are slow; one big file is fast. JPEG doesn't compress further, so we skip gzip.

In [ ]:
CACHE_TAR = f'{DRIVE_CACHE}/{CACHE_NAME}_{SUBSET}.tar'

def n_cached():
    return sum(os.path.exists(cache_path(r)) for r in data.rel_path)

if n_cached() < len(data) and os.path.exists(CACHE_TAR):
    print('Restoring cache from Drive...')
    subprocess.run(['tar', '-xf', CACHE_TAR, '-C', LOCAL_DATA], check=True)

todo = [(r, n) for r, n in zip(data.rel_path, data.n_frames) if not os.path.exists(cache_path(r))]
print(f'{len(data) - len(todo)} cached, {len(todo)} to build')
if todo:
    t0 = time.perf_counter()
    with ThreadPoolExecutor(max_workers=min(8, 2 * os.cpu_count())) as ex:
        status = list(tqdm(ex.map(lambda a: build_clip_cache(*a), todo), total=len(todo), desc='Caching'))
    el = time.perf_counter() - t0
    print(pd.Series(status).value_counts().to_string())
    print(f'Built {len(todo)} clips in {el / 60:.1f} min ({el / len(todo) * 1000:.0f} ms/clip)')
    subprocess.run(['tar', '-cf', CACHE_TAR, '-C', LOCAL_DATA, CACHE_NAME], check=True)
    print(f'Backed up to {CACHE_TAR} ({os.path.getsize(CACHE_TAR) / 1e9:.2f} GB)')

missing = [r for r in data.rel_path if not os.path.exists(cache_path(r))]
assert not missing, f'{len(missing)} clips failed to cache, e.g. {missing[:3]}'
print(f'✅ All {len(data)} clips cached')

`ok_padded` in the status counts means a clip had fewer decodable frames than the header claimed and some indices were filled with the nearest real frame. A handful is normal. If it's a large fraction, tell me, because it would change how we interpret those clips.

> When you switch `SUBSET = 'full'` later, the mini clips are already cached and are skipped automatically.

### 3.4 Normalization statistics: ImageNet, Kinetics, or our own?

Normalizing inputs to roughly zero mean and unit variance per channel keeps activations well scaled at the start of training. Which statistics should we use?

- **Pretrained model:** always use the statistics it was **trained with**. For torchvision's R(2+1)D that's **Kinetics**, not ImageNet (the assignment's Section 5.2 uses ImageNet). Wrong statistics shift every input the pretrained filters see.
- **Training from scratch (C3D):** any reasonable statistics work. Let's measure UCF101's own and compare.

In [ ]:
stat_rows = data[data.phase == 'train'].sample(min(200, (data.phase == 'train').sum()), random_state=SEED)
pix = np.concatenate([load_cached(cache_path(r), which=[0, 10, 21, 31]).reshape(-1, 3)
                      for r in tqdm(stat_rows.rel_path, desc='Stats')]).astype(np.float64) / 255
ucf_mean, ucf_std = pix.mean(0), pix.std(0)

stats = pd.DataFrame({'UCF101 (measured)': [*ucf_mean, *ucf_std],
                      'ImageNet': [0.485, 0.456, 0.406, 0.229, 0.224, 0.225],
                      'Kinetics': [0.43216, 0.394666, 0.37645, 0.22803, 0.22145, 0.216989]},
                     index=['mean_R', 'mean_G', 'mean_B', 'std_R', 'std_G', 'std_B']).round(3)
stats

**Decision.** If the measured UCF101 statistics are within a few hundredths of ImageNet's (they usually are), we'll use **ImageNet stats for models trained from scratch**, which matches the assignment and keeps results comparable, and **Kinetics stats for pretrained R(2+1)D**. The Dataset below takes `norm=` as a parameter so each model gets the right one.

### 3.5 Write `utils/dataset.py`: segment sampling, clip-consistent augmentation, normalization

This becomes part of your code submission (`utils/dataset.py` in Section 3.4 of the assignment). Key ideas, each explained in the code:

1. **Segment sampling.** Split the K=32 cached frames into T=16 equal segments. **Train:** a random frame per segment (temporal jitter). **Eval:** a fixed frame per segment, so validation and test are deterministic.
2. **Clip-consistent augmentation.** The crop position and the flip are chosen **once per clip** and applied to **every frame**. If each frame got its own random crop, the image would jump around between frames, and the 3D CNN would learn that **fake motion**.
3. **Transform parameters are returned**, so that in Step 4 the optical flow for the same clip gets the *same* crop and flip. Note that flipping a flow field also requires negating its horizontal component.
4. **Python's `random` (not NumPy's)** inside the Dataset, because PyTorch reseeds Python's RNG in each DataLoader worker. With NumPy's global RNG, older PyTorch versions give every worker the *same* random sequence, so all workers produce identical "random" augmentations.
5. An assignment-compatible **`.samples`** list of `(rel_path, label)`, so the `PairDataset` (Part 3) and `MultimodalDataset` (Part 4) templates work unchanged.

In [ ]:
%%writefile /content/drive/MyDrive/videosense/code/utils/dataset.py
"""VideoSense datasets: cached-frame loading, segment sampling, clip-consistent augmentation."""
import os
import random

import cv2
import numpy as np
import torch
from torch.utils.data import Dataset

NORM_STATS = {
    'imagenet': ([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),           # C3D from scratch
    'kinetics': ([0.43216, 0.394666, 0.37645], [0.22803, 0.22145, 0.216989]),  # torchvision video models
}


def decode_frames(data, offsets, which):
    """Decode selected JPEG frames from packed bytes -> (n, H, W, 3) uint8 RGB."""
    return np.stack([cv2.cvtColor(cv2.imdecode(data[offsets[i]:offsets[i + 1]], cv2.IMREAD_COLOR),
                                  cv2.COLOR_BGR2RGB) for i in which])


def load_cached_frames(path, which=None):
    """Decode frames of a cached clip (all frames if which is None)."""
    z = np.load(path)
    data, off = z['data'], z['offsets']
    return decode_frames(data, off, range(len(off) - 1) if which is None else which)


def segment_indices(k, t, train):
    """Pick t of k stored frames: one per equal-width segment.

    train=True  -> random frame inside each segment (temporal augmentation)
    train=False -> fixed frame per segment (deterministic evaluation)
    """
    edges = np.linspace(0, k, t + 1)
    lo = np.floor(edges[:-1]).astype(int)
    hi = np.maximum(np.floor(edges[1:]).astype(int), lo + 1)
    if train:
        return [random.randrange(a, b) for a, b in zip(lo, hi)]
    return ((lo + hi - 1) // 2).tolist()


class ClipTransform:
    """Spatial crop + optional horizontal flip, identical for every frame of a clip."""

    def __init__(self, crop=112, train=False, hflip_p=0.5):
        self.crop, self.train, self.hflip_p = crop, train, hflip_p

    def sample_params(self, h, w):
        c = self.crop
        if self.train:
            return {'y': random.randint(0, h - c), 'x': random.randint(0, w - c),
                    'flip': random.random() < self.hflip_p}
        return {'y': (h - c) // 2, 'x': (w - c) // 2, 'flip': False}

    def __call__(self, frames, params=None):
        """frames: (T, H, W, C) array. Returns (cropped frames, params used)."""
        params = params or self.sample_params(*frames.shape[1:3])
        y, x, c = params['y'], params['x'], self.crop
        out = frames[:, y:y + c, x:x + c]
        if params['flip']:
            out = out[:, :, ::-1]
        return np.ascontiguousarray(out), params


def to_normalized_tensor(frames, norm='imagenet'):
    """(T, H, W, 3) uint8 -> (3, T, H, W) float32, channel-normalized."""
    mean, std = (torch.tensor(v).view(3, 1, 1, 1) for v in NORM_STATS[norm])
    x = torch.from_numpy(frames).permute(3, 0, 1, 2).float().div_(255.0)
    return (x - mean) / std


def denormalize(clip, norm='imagenet'):
    """(3, T, H, W) normalized tensor -> (T, H, W, 3) float array in [0, 1] for plotting."""
    mean, std = (torch.tensor(v).view(3, 1, 1, 1) for v in NORM_STATS[norm])
    return (clip * std + mean).clamp(0, 1).permute(1, 2, 3, 0).numpy()


class UCF101Clips(Dataset):
    """RGB clips from the frame cache.

    df              : manifest rows with columns rel_path, label
    cache_dir       : folder with <Class>/<video>.npz files
    temporal_jitter : random frame per segment (defaults to train)
    Returns   : (clip (3, T, crop, crop) float32, label int)
    """

    def __init__(self, df, cache_dir, num_frames=16, train=False, crop=112, norm='imagenet',
                 temporal_jitter=None):
        self.df = df.reset_index(drop=True)
        self.cache_dir, self.num_frames, self.train, self.norm = cache_dir, num_frames, train, norm
        # temporal_jitter defaults to `train`; set it alone (train=False) for temporal test-time augmentation
        self.temporal_jitter = train if temporal_jitter is None else temporal_jitter
        self.transform = ClipTransform(crop=crop, train=train)
        self.labels = self.df.label.astype(int).tolist()
        self.samples = list(zip(self.df.rel_path, self.labels))      # assignment-compatible
        self.class_names = (self.df.drop_duplicates('label').sort_values('label').class_name.tolist())

    def __len__(self):
        return len(self.df)

    def path(self, i):
        return os.path.join(self.cache_dir, self.df.rel_path[i][:-4] + '.npz')

    def get(self, i):
        """Full output: clip, label, and the sampling/transform parameters used."""
        z = np.load(self.path(i))                      # one file read per clip
        data, off = z['data'], z['offsets']
        which = segment_indices(len(off) - 1, self.num_frames, self.temporal_jitter)
        frames, params = self.transform(decode_frames(data, off, which))
        params['frame_idx'] = which
        return to_normalized_tensor(frames, self.norm), self.labels[i], params

    def __getitem__(self, i):
        clip, label, _ = self.get(i)
        return clip, label


### 3.6 Sanity checks — shapes, values, labels, and augmentation

In [ ]:
import importlib
if CODE_DIR not in sys.path:
    sys.path.insert(0, CODE_DIR)
import utils.dataset as vds
importlib.reload(vds)

phase_df = {p: data[data.phase == p] for p in ('train', 'val', 'test')}
ds = {p: vds.UCF101Clips(d, LOCAL_CACHE, num_frames=T, train=(p == 'train'), crop=CROP, norm='imagenet')
      for p, d in phase_df.items()}
n_classes = data.label.nunique()

for p, d in ds.items():
    clip, label = d[0]
    print(f'{p:>5}: {len(d):5d} clips | clip {tuple(clip.shape)} {clip.dtype} | '
          f'range [{clip.min():.2f}, {clip.max():.2f}] | label {label}')

clip, _ = ds['train'][0]
assert clip.shape == (3, T, CROP, CROP) and clip.dtype == torch.float32
assert torch.isfinite(clip).all()
assert set(data.label) == set(range(n_classes)), 'labels must be 0..C-1'

# Determinism: eval must give identical tensors every call; train should not
assert torch.equal(ds['val'][0][0], ds['val'][0][0]), 'val is not deterministic!'
print('Train calls differ (augmentation active):', not torch.equal(ds['train'][0][0], ds['train'][0][0]))

# Normalization check over a few clips: mean should be near 0, std near 1
batch = torch.stack([ds['val'][i][0] for i in range(min(16, len(ds['val'])))])
print('Per-channel mean:', batch.mean(dim=(0, 2, 3, 4)).numpy().round(2),
      '| std:', batch.std(dim=(0, 2, 3, 4)).numpy().round(2))
print('✅ Dataset checks passed')

In [ ]:
# See what the model sees: two training augmentations vs the deterministic eval view of the same clip
i = 0
views = [('train view A', *ds['train'].get(i)), ('train view B', *ds['train'].get(i))]
eval_ds = vds.UCF101Clips(phase_df['train'], LOCAL_CACHE, num_frames=T, train=False, crop=CROP)
views.append(('eval view', *eval_ds.get(i)))

show_t = [0, 5, 10, 15]
fig, axes = plt.subplots(len(views), len(show_t), figsize=(12, 9))
for r, (name, clip, label, params) in enumerate(views):
    frames = vds.denormalize(clip)
    for c, t in enumerate(show_t):
        axes[r, c].imshow(frames[t]); axes[r, c].axis('off')
        axes[r, c].set_title(f'{name}\ncached frame {params["frame_idx"][t]}' if c == 0
                             else f'cached frame {params["frame_idx"][t]}', fontsize=9)
    axes[r, 0].text(-5, 56, f'crop y={params["y"]},x={params["x"]}\nflip={params["flip"]}',
                    ha='right', va='center', fontsize=8, transform=axes[r, 0].transData)
plt.suptitle(f'{phase_df["train"].class_name.iloc[i]}: same clip, three views')
plt.tight_layout(); save_fig('preprocess_augmentation_views'); plt.show()

**What to check in the figure:** within each row, the crop and flip are identical across frames (no artificial jitter). Between the two train rows, the crop, flip, or chosen frames differ (that's the augmentation). The eval row is the centered, unflipped view and is identical every time you run it.

### 3.7 Is data loading fast enough?

The GPU can only train as fast as the DataLoader feeds it. We time batches with different numbers of worker processes and compare against Step 2's per-clip decode cost.

In [ ]:
from torch.utils.data import DataLoader

def time_loader(dataset, workers, n_batches=15, bs=8):
    dl = DataLoader(dataset, batch_size=bs, shuffle=True, num_workers=workers,
                    persistent_workers=workers > 0)
    it = iter(dl); next(it)                               # warm-up (worker start-up)
    t0 = time.perf_counter()
    for _ in range(n_batches):
        try:
            next(it)
        except StopIteration:
            break
    return (time.perf_counter() - t0) / (n_batches * bs) * 1000

for w in sorted({0, 2, os.cpu_count()}):
    print(f'num_workers={w}: {time_loader(ds["train"], w):.1f} ms/clip')
if 'timing' in globals():
    print(f'(Step 2, raw video: seek {timing["seek"]*1000:.0f} ms/clip, sequential {timing["sequential"]*1000:.0f} ms/clip)')

**Interpretation.** A C3D forward+backward pass on a T4 takes very roughly 10–20 ms per clip. If loading is below that with your chosen `num_workers`, the GPU won't be starved. On Colab (2 CPUs), `num_workers=2` is usually the sweet spot. More workers than CPUs doesn't help.

---
### ✅ Step 3 checklist
- [ ] Train / val / test split by group, saved as `splits_{mini,full}.csv`
- [ ] Frame cache built, PSNR acceptable, tar backed up to Drive
- [ ] Normalization stats compared; ImageNet for scratch, Kinetics for pretrained
- [ ] `utils/dataset.py` written; all sanity asserts pass
- [ ] Augmentation figure looks right (consistent across frames)
- [ ] Loader speed measured

**Next — Step 4 (Optical flow):** compute Farneback flow from the **original consecutive frames** (not the JPEG cache: JPEG artifacts would show up as fake motion), at the same K positions as the RGB cache so the two streams stay aligned. Then visualize it (Part 1 Exercise 5) and extend the Dataset to return flow with matching crop and flip.

## Step 4 — Optical flow: compute, understand, cache, and load

**Goal:** build the second modality for the two-stream network (Part 4), and complete Part 1 (Exercises 3 and 5, Reflection Q1–Q3).

**How this step differs from the assignment's template, and why**

| Template (Section 5.3 / 8.3) | This notebook | Reason |
|---|---|---|
| Flow from frames resized to 112×112 | Flow at **128×171**, the same grid as the RGB cache | Same aspect ratio, and the RGB crop can be applied to flow pixel-for-pixel |
| Its own 16 frame positions | The **same 32 positions** stored in the RGB cache (`src_idx`) | RGB frame *t* and flow frame *t* describe the same moment |
| Saved as float `.npy` (~2.8 MB/clip) | Clipped to ±B px, **uint8 JPEG** (~10× smaller) | The full dataset would not fit on a free Drive otherwise |
| Normalized **per clip** by its max | Normalized by a **fixed global bound** B | Per-clip scaling erases absolute speed: a slow walk and a sprint would look equally fast |
| No camera handling | Optional **camera-motion compensation** | Reflection Q2 |

Flow is computed from the **original decoded video frames**, not the JPEG cache. Compression artifacts differ from frame to frame, and Farneback would report them as motion.

> **CPU runtime.** In a new session, run 1.1–1.4, 2.0, 3.0–3.1 and 3.3 first (videos, paths, splits, RGB cache).

### 4.0 Write `utils/optical_flow.py`

Flow computation, quantization, storage and visualization live in a module. That's the `utils/optical_flow.py` file the assignment's code structure asks for, and the Dataset in 4.6 imports from it.

In [ ]:
%%writefile /content/drive/MyDrive/videosense/code/utils/optical_flow.py
"""Dense optical flow (Farneback): computation, quantized JPEG storage, visualization."""
import cv2
import numpy as np

# Farneback parameters from the assignment (Section 5.3)
FARNEBACK = dict(pyr_scale=0.5, levels=3, winsize=15, iterations=3,
                 poly_n=5, poly_sigma=1.2, flags=0)


def read_gray(video_path, wanted, size):
    """Sequentially decode the wanted frame indices as resized grayscale. size=(w, h)."""
    want, got = set(int(i) for i in wanted), {}
    cap, i = cv2.VideoCapture(video_path), 0
    while len(got) < len(want) and cap.grab():
        if i in want:
            ok, f = cap.retrieve()
            if ok:
                got[i] = cv2.resize(cv2.cvtColor(f, cv2.COLOR_BGR2GRAY), size,
                                    interpolation=cv2.INTER_AREA)
        i += 1
    cap.release()
    return got


def compute_flows(video_path, src_idx, size):
    """Flow from frame j to j+1 for every j in src_idx -> ((K, h, w, 2) float32, #zero-filled).

    At the last frame of a clip, (j-1 -> j) is used instead. If neither pair is
    decodable, the flow is zero-filled and counted.
    """
    wanted = set()
    for j in src_idx:
        wanted |= {max(int(j) - 1, 0), int(j), int(j) + 1}
    got = read_gray(video_path, wanted, size)
    flows, n_zero = [], 0
    for j in src_idx:
        j = int(j)
        for a, b in ((j, j + 1), (j - 1, j)):
            if a in got and b in got and a != b:
                flows.append(cv2.calcOpticalFlowFarneback(got[a], got[b], None, **FARNEBACK))
                break
        else:
            flows.append(np.zeros((size[1], size[0], 2), np.float32))
            n_zero += 1
    return np.stack(flows), n_zero


def quantize_flow(flow, bound):
    """px -> uint8. 0 px maps exactly to 128; +-bound px maps to 255 / 1 (values beyond are clipped)."""
    return np.clip(np.round(flow / bound * 127) + 128, 1, 255).astype(np.uint8)


def dequantize_flow(q):
    """uint8 -> float32 in [-1, 1], in units of the bound (multiply by bound for pixels)."""
    return (q.astype(np.float32) - 128.0) / 127.0


def save_flow_npz(path, flows, bound, src_idx, jpeg_q=95):
    """Store K flow fields as 2K grayscale JPEGs (u0, v0, u1, v1, ...) in one .npz."""
    q = quantize_flow(flows, bound)
    jpgs = [cv2.imencode('.jpg', np.ascontiguousarray(q[k, :, :, c]),
                         [cv2.IMWRITE_JPEG_QUALITY, jpeg_q])[1].ravel()
            for k in range(len(q)) for c in range(2)]
    offsets = np.concatenate([[0], np.cumsum([len(j) for j in jpgs])])
    with open(path, 'wb') as fh:
        np.savez(fh, data=np.concatenate(jpgs), offsets=offsets,
                 src_idx=np.asarray(src_idx), bound=np.float32(bound))


def decode_flow(data, offsets, which):
    """Decode selected flow frames -> (n, h, w, 2) float32 in [-1, 1]."""
    out = []
    for k in which:
        u = cv2.imdecode(data[offsets[2 * k]:offsets[2 * k + 1]], cv2.IMREAD_GRAYSCALE)
        v = cv2.imdecode(data[offsets[2 * k + 1]:offsets[2 * k + 2]], cv2.IMREAD_GRAYSCALE)
        out.append(np.stack([u, v], axis=-1))
    return dequantize_flow(np.stack(out))


def load_flow(path, which=None):
    z = np.load(path)
    data, off = z['data'], z['offsets']
    k = (len(off) - 1) // 2
    return decode_flow(data, off, range(k) if which is None else which), float(z['bound'])


def flow_to_hsv(flow, max_mag=None):
    """(h, w, 2) flow -> RGB image: hue = direction, brightness = speed.

    Pass a shared max_mag when comparing several frames or clips; per-image
    scaling makes a nearly static frame look as 'bright' as a fast one.
    """
    u = np.ascontiguousarray(flow[..., 0], np.float32)
    v = np.ascontiguousarray(flow[..., 1], np.float32)
    mag, ang = cv2.cartToPolar(u, v)
    hsv = np.zeros((*flow.shape[:2], 3), np.uint8)
    hsv[..., 0] = (ang * 90 / np.pi).astype(np.uint8)          # OpenCV hue range is 0..179
    hsv[..., 1] = 255
    m = max_mag if max_mag else mag.max() + 1e-6
    hsv[..., 2] = np.clip(mag / m * 255, 0, 255).astype(np.uint8)
    return cv2.cvtColor(hsv, cv2.COLOR_HSV2RGB)


def color_wheel(size=101):
    """Legend for flow_to_hsv: the color of each motion direction."""
    yy, xx = np.mgrid[-1:1:size * 1j, -1:1:size * 1j].astype(np.float32)
    img = flow_to_hsv(np.stack([xx, yy], -1), max_mag=1.0)
    img[(xx ** 2 + yy ** 2) > 1] = 255
    return img


In [ ]:
import importlib, json
if CODE_DIR not in sys.path:
    sys.path.insert(0, CODE_DIR)
import utils.optical_flow as oflow
importlib.reload(oflow)

FLOW_SIZE   = (CACHE_W, CACHE_H)       # cv2 wants (w, h)
FLOW_JPEG_Q = 95                       # higher than RGB: flow is smooth, artifacts would look like motion

def video_path(rel):
    return f'{VIDEO_DIR}/{rel}'

def src_idx_of(rel):
    """Frame positions used by the RGB cache, so both modalities stay aligned."""
    return np.load(cache_path(rel))['src_idx']
print('optical_flow module loaded')

### 4.1 Measure flow on a sample: how fast is it, and how big do the vectors get?

We need two numbers before building the cache:

1. **Cost per frame pair**, to estimate how long the full build takes.
2. **The range of flow values**, to choose the clipping bound B for uint8 storage. Too small a bound clips real fast motion; too large wastes the 256 levels on values that never occur (coarser steps for everyday motion). We take the 99.9th percentile of |u|, |v| and keep it between 10 and 30 px. Published two-stream pipelines typically use about 20 px at similar resolutions.

We also collect per-class statistics for 4.4.

In [ ]:
probe_clips = (data[data.phase == 'train'].sample(frac=1, random_state=SEED)
               .groupby('class_name').head(6))
abs_vals, rows, pair_time, n_pairs = [], [], 0.0, 0
for r in tqdm(probe_clips.itertuples(), total=len(probe_clips), desc='Probing flow'):
    src = src_idx_of(r.rel_path)[::4]                         # 8 of the 32 positions
    t0 = time.perf_counter()
    fl, _ = oflow.compute_flows(video_path(r.rel_path), src, FLOW_SIZE)
    pair_time += time.perf_counter() - t0; n_pairs += len(src)
    abs_vals.append(np.abs(fl[::2, ::4, ::4]).ravel())        # subsample pixels to save memory
    comp = fl - np.median(fl, axis=(1, 2), keepdims=True)    # camera-compensated (see 4.3)
    rows.append(dict(class_name=r.class_name,
                     mag=np.linalg.norm(fl, axis=-1).mean(),
                     mag_comp=np.linalg.norm(comp, axis=-1).mean(),
                     horiz_vs_vert=np.abs(comp[..., 0]).mean() / (np.abs(comp[..., 1]).mean() + 1e-6)))
abs_vals = np.concatenate(abs_vals)
flow_stats = pd.DataFrame(rows)

pct = np.percentile(abs_vals, [50, 95, 99, 99.9])
FLOW_BOUND = float(np.clip(np.ceil(pct[3]), 10, 30))
ms_pair = pair_time / n_pairs * 1000
print('|u|,|v| percentiles (px): p50={:.2f}  p95={:.2f}  p99={:.2f}  p99.9={:.2f}'.format(*pct))
print(f'Chosen bound B = {FLOW_BOUND:.0f} px -> clips {(abs_vals > FLOW_BOUND).mean():.3%} of values; '
      f'quantization step {2 * FLOW_BOUND / 254:.2f} px')
print(f'Cost: {ms_pair:.1f} ms per frame pair (incl. decoding) -> '
      f'~{ms_pair * CACHE_K * len(data) / 60000 / os.cpu_count():.0f} min '
      f'estimated for {len(data)} clips ({SUBSET})')

### 4.2 Visualize flow (Part 1, Exercise 5)

For 4 consecutive frame pairs we show the RGB frame, the **horizontal component u** (red = moving right, blue = moving left), the **vertical component v** (red = moving down, since image y points down), and the **HSV encoding**: hue = direction (see the color wheel), brightness = speed.

All panels share one scale, so brightness is comparable across frames.

In [ ]:
FLOW_DEMO_CLASS = 'PullUps'      # change to any class to explore
row = data[(data.class_name == FLOW_DEMO_CLASS) & (data.phase == 'train')].iloc[0]
start = int(row.n_frames) // 2
pairs = list(range(start, start + 4))
fl, _ = oflow.compute_flows(video_path(row.rel_path), pairs, FLOW_SIZE)
rgb = read_frames(row.rel_path, pairs)                         # helper from Step 2.7
lim = max(np.percentile(np.abs(fl), 99), 1e-3)
mag_max = np.percentile(np.linalg.norm(fl, axis=-1), 99)

fig, axes = plt.subplots(4, 5, figsize=(18, 12), layout='constrained',
                         gridspec_kw={'width_ratios': [1, 1, 1, 1, 0.55]})
for c in range(4):
    axes[0, c].imshow(rgb[c]); axes[0, c].set_title(f'frame {pairs[c]} -> {pairs[c] + 1}')
    im_u = axes[1, c].imshow(fl[c, ..., 0], cmap='coolwarm', vmin=-lim, vmax=lim)
    im_v = axes[2, c].imshow(fl[c, ..., 1], cmap='coolwarm', vmin=-lim, vmax=lim)
    axes[3, c].imshow(oflow.flow_to_hsv(fl[c], max_mag=mag_max))
for r_, name in enumerate(['RGB', 'u (horizontal)', 'v (vertical)', 'HSV (dir + speed)']):
    axes[r_, 0].set_ylabel(name, fontsize=12)
for ax in axes[:, :4].flat:
    ax.set_xticks([]); ax.set_yticks([])
for ax in axes[:3, 4]:
    ax.axis('off')
fig.colorbar(im_u, ax=axes[1:3, 4], fraction=0.5, label='pixels / frame')
axes[3, 4].imshow(oflow.color_wheel()); axes[3, 4].set_title('direction key'); axes[3, 4].axis('off')
plt.suptitle(f'{row.class_name}: optical flow for 4 consecutive frame pairs ({row.rel_path})')
save_fig('part1_ex5_optical_flow'); plt.show()

**What to notice:** flow is concentrated on the moving body parts, while static background is dark. Faint speckles in the background are noise from lighting changes and compression, a reminder that flow is an *estimate* built on the assumption that a pixel keeps its brightness as it moves. Weakly textured regions (plain walls, sky) are where that assumption carries the least information, so they're noisiest.

### 4.3 Camera motion vs. subject motion (Reflection Q2)

When the camera pans, **every pixel** moves, so the flow field is dominated by the camera rather than the person. A simple estimate of camera motion is the **median flow vector of the frame**. If most pixels are background, the median follows the background. Subtracting it leaves (approximately) the motion *relative to the scene*.

We use the class with the strongest border motion from Step 2.8 (if available), since that's where camera motion is most likely.

**Limitations to mention in your answer:** the median models only *translation* (a pan or tilt). Zoom, rotation, or perspective change need a geometric model fit to the flow, for example an affine transform or homography estimated with RANSAC (`cv2.findHomography`), which treats the moving person as outliers. This is how *improved dense trajectories* (Wang & Schmid, 2013) handle camera motion. It also fails when the subject fills most of the frame, because then the median follows the subject.

In [ ]:
cam_class = (summary.border_center.idxmax() if 'summary' in globals() else 'Surfing')
row = data[(data.class_name == cam_class) & (data.phase == 'train')].iloc[0]
src = src_idx_of(row.rel_path)
fl, _ = oflow.compute_flows(video_path(row.rel_path), src, FLOW_SIZE)
med = np.median(fl, axis=(1, 2))                               # (K, 2) estimated camera motion
comp = fl - med[:, None, None, :]
show = np.linspace(0, len(src) - 1, 4, dtype=int)
mag_max = np.percentile(np.linalg.norm(fl, axis=-1), 99)

fig = plt.figure(figsize=(18, 9))
gs = fig.add_gridspec(3, 4, height_ratios=[1, 1, 0.8])
for c, k in enumerate(show):
    for r_, (f_, name) in enumerate([(fl, 'raw flow'), (comp, 'median subtracted')]):
        ax = fig.add_subplot(gs[r_, c])
        ax.imshow(oflow.flow_to_hsv(f_[k], max_mag=mag_max)); ax.set_xticks([]); ax.set_yticks([])
        if c == 0:
            ax.set_ylabel(name, fontsize=12)
        if r_ == 0:
            ax.set_title(f'source frame {src[k]}')
ax = fig.add_subplot(gs[2, :])
ax.plot(src, med[:, 0], label='median u (horizontal)'); ax.plot(src, med[:, 1], label='median v (vertical)')
ax.axhline(0, color='k', lw=0.5); ax.set(xlabel='source frame', ylabel='px / frame',
                                         title='Estimated camera motion over the clip'); ax.legend()
plt.suptitle(f'{cam_class}: camera-motion compensation by median subtraction')
plt.tight_layout(); save_fig('flow_camera_compensation'); plt.show()

print(f'Mean |flow| raw = {np.linalg.norm(fl, axis=-1).mean():.2f} px, '
      f'after compensation = {np.linalg.norm(comp, axis=-1).mean():.2f} px')

### 4.4 What does flow reveal about each action? (Reflection Q3)

UCF101 has no plain "Running" class, so for the walking-vs-running comparison we use **WalkingWithDog** against **LongJump**, whose clips include a sprinting run-up. Change `CMP_CLASSES` if you'd like a different pair. Both clips are shown on the **same** speed scale.

We also summarize motion per mini class, using the sample from 4.1:
- **Mean speed** before and after camera compensation (a large drop means much of the flow was camera motion).
- **Horizontal / vertical ratio**: above 1 means mostly sideways motion (walking, fencing), below 1 means mostly up-and-down (pull-ups, push-ups).

In [ ]:
CMP_CLASSES = ['WalkingWithDog', 'LongJump']
cmp_flows = {}
for cls in CMP_CLASSES:
    r = full[(full.class_name == cls) & (full.phase == 'train')].iloc[0]
    mid = int(r.n_frames) // 2
    pairs = np.linspace(mid - 20, mid + 20, 4, dtype=int).clip(0, int(r.n_frames) - 2)
    cmp_flows[cls] = (r, pairs, oflow.compute_flows(video_path(r.rel_path), pairs, FLOW_SIZE)[0])
shared_max = max(np.percentile(np.linalg.norm(f, axis=-1), 99) for _, _, f in cmp_flows.values())

fig, axes = plt.subplots(len(CMP_CLASSES) * 2, 4, figsize=(16, 4.2 * len(CMP_CLASSES)))
for i, (cls, (r, pairs, f_)) in enumerate(cmp_flows.items()):
    rgb = read_frames(r.rel_path, pairs)
    for c in range(4):
        axes[2 * i, c].imshow(rgb[min(c, len(rgb) - 1)])
        axes[2 * i + 1, c].imshow(oflow.flow_to_hsv(f_[c], max_mag=shared_max))
        for ax in (axes[2 * i, c], axes[2 * i + 1, c]):
            ax.set_xticks([]); ax.set_yticks([])
    axes[2 * i, 0].set_ylabel(cls, fontsize=12); axes[2 * i + 1, 0].set_ylabel('flow (shared scale)')
plt.tight_layout(); save_fig('reflection_q3_walk_vs_run_flow'); plt.show()

plt.figure(figsize=(8, 4))
for cls, (_, _, f_) in cmp_flows.items():
    plt.hist(np.linalg.norm(f_, axis=-1).ravel(), bins=100, range=(0, shared_max * 1.5),
             histtype='step', density=True, lw=2, label=cls)
plt.yscale('log'); plt.xlabel('flow magnitude (px / frame)'); plt.ylabel('density (log)')
plt.title('Speed distribution'); plt.legend(); save_fig('reflection_q3_speed_hist'); plt.show()

per_class = flow_stats.groupby('class_name')[['mag', 'mag_comp', 'horiz_vs_vert']].mean()
per_class['camera_share'] = 1 - per_class.mag_comp / per_class.mag
per_class.sort_values('mag').round(2)

**Interpretation for Q3.** The faster action shows a heavier tail of large flow vectors and brighter, more coherent regions on the limbs. Walking produces moderate, mostly horizontal motion with a regular rhythm. For the network this means **speed** and **direction** carry class information that a single RGB frame can't. Per-clip max normalization (the template's approach) would rescale both clips to the same peak and **throw that speed difference away**, which is why we normalize by the fixed bound B instead.

### 4.5 Build (or restore) the flow cache (Part 1, Exercise 3)

The same pattern as the RGB cache: restore the tar from Drive if it exists, otherwise build with threads, skipping finished clips, with atomic writes. Then back up one tar. First, we test one clip and measure how much accuracy uint8 + JPEG storage costs, as **end-point error (EPE)**: the average length of the difference vector between stored and original flow.

In [ ]:
FLOW_NAME  = f'flow_k{CACHE_K}_{CACHE_H}x{CACHE_W}_b{FLOW_BOUND:.0f}_q{FLOW_JPEG_Q}'
LOCAL_FLOW = f'{LOCAL_DATA}/{FLOW_NAME}'
FLOW_TAR   = f'{DRIVE_CACHE}/{FLOW_NAME}_{SUBSET}.tar'
os.makedirs(LOCAL_FLOW, exist_ok=True)

def flow_path(rel):
    return f'{LOCAL_FLOW}/{rel[:-4]}.npz'

def build_flow_cache(rel):
    out = flow_path(rel)
    if os.path.exists(out):
        return 'skip'
    src = src_idx_of(rel)
    flows, n_zero = oflow.compute_flows(video_path(rel), src, FLOW_SIZE)
    os.makedirs(os.path.dirname(out), exist_ok=True)
    oflow.save_flow_npz(out + '.tmp', flows, FLOW_BOUND, src, FLOW_JPEG_Q)
    os.replace(out + '.tmp', out)
    return 'ok' if n_zero == 0 else 'ok_zero_pairs'

# --- test one clip: storage cost vs accuracy
r = data[data.phase == 'train'].iloc[0]
exact, _ = oflow.compute_flows(video_path(r.rel_path), src_idx_of(r.rel_path), FLOW_SIZE)
print(build_flow_cache(r.rel_path))
stored, bound = oflow.load_flow(flow_path(r.rel_path))
inside = (np.abs(exact) <= FLOW_BOUND).all(-1)                 # ignore clipped vectors
epe = np.linalg.norm(stored * bound - exact, axis=-1)[inside].mean()
size_kb = os.path.getsize(flow_path(r.rel_path)) / 1e3
f16_kb = exact.size * 2 / 1e3
print(f'Stored {stored.shape} | {size_kb:.0f} KB vs float16 {f16_kb:.0f} KB ({f16_kb / size_kb:.1f}x smaller) | '
      f'EPE {epe:.3f} px (typical motion p50 = {pct[0]:.2f} px)')

In [ ]:
def n_flow_cached():
    return sum(os.path.exists(flow_path(r)) for r in data.rel_path)

if n_flow_cached() < len(data) and os.path.exists(FLOW_TAR):
    print('Restoring flow cache from Drive...')
    subprocess.run(['tar', '-xf', FLOW_TAR, '-C', LOCAL_DATA], check=True)

todo = [r for r in data.rel_path if not os.path.exists(flow_path(r))]
print(f'{len(data) - len(todo)} cached, {len(todo)} to build')
if todo:
    t0 = time.perf_counter()
    with ThreadPoolExecutor(max_workers=min(8, 2 * os.cpu_count())) as ex:
        status = list(tqdm(ex.map(build_flow_cache, todo), total=len(todo), desc='Flow cache'))
    el = time.perf_counter() - t0
    print(pd.Series(status).value_counts().to_string())
    print(f'Built {len(todo)} clips in {el / 60:.1f} min ({el / len(todo):.2f} s/clip)')
    subprocess.run(['tar', '-cf', FLOW_TAR, '-C', LOCAL_DATA, FLOW_NAME], check=True)
    print(f'Backed up to {FLOW_TAR} ({os.path.getsize(FLOW_TAR) / 1e9:.2f} GB)')

missing = [r for r in data.rel_path if not os.path.exists(flow_path(r))]
assert not missing, f'{len(missing)} clips missing flow, e.g. {missing[:3]}'

# Save preprocessing settings so later steps (and the README) use the same values
cfg = dict(T=T, CACHE_K=CACHE_K, CACHE_H=CACHE_H, CACHE_W=CACHE_W, CROP=CROP, JPEG_Q=JPEG_Q,
           CACHE_NAME=CACHE_NAME, FLOW_NAME=FLOW_NAME, FLOW_BOUND=FLOW_BOUND,
           FLOW_JPEG_Q=FLOW_JPEG_Q, VAL_GROUPS=sorted(VAL_GROUPS), SEED=SEED)
json.dump(cfg, open(f'{DRIVE_META}/preprocess_config.json', 'w'), indent=2)
print(f'✅ Flow cached for all {len(data)} clips; config saved')

**Timing note:** flow is the slowest preprocessing step. On Colab's 2 CPUs the mini subset should take minutes; the full dataset can take a few hours. For the full build, start it in a session you can leave running. It resumes where it stopped if Colab disconnects.

### 4.6 Write `utils/flow_dataset.py`: flow clips aligned with RGB

- **`FlowClips`** returns `(2, T, 112, 112)` flow in units of B (so values lie in [-1, 1]), with the same segment sampling and crop as RGB. Option `camera_comp=True` subtracts the per-frame median (4.3) *before* cropping, using the whole frame to estimate camera motion.
- **`MultimodalClips`** returns `(rgb, flow, label)`. It samples **one** set of frame indices, crop, and flip, and applies it to **both** streams. Without this, random augmentation would crop the RGB at the top-left and the flow at the bottom-right, and the fusion model would be combining two different regions.
- **Flipping flow is not just mirroring the image.** When the picture is mirrored, motion to the right becomes motion to the left, so **u must also change sign**. v (vertical) is unchanged. Forgetting this gives the network flow fields that contradict the RGB.

In [ ]:
%%writefile /content/drive/MyDrive/videosense/code/utils/flow_dataset.py
"""Flow and RGB+flow datasets that share frame sampling, crop and flip."""
import os

import numpy as np
import torch
from torch.utils.data import Dataset

from utils.dataset import ClipTransform, segment_indices
from utils.optical_flow import decode_flow


class FlowClips(Dataset):
    """Optical-flow clips from the flow cache -> ((2, T, crop, crop) float32, label)."""

    def __init__(self, df, flow_dir, num_frames=16, train=False, crop=112, camera_comp=False,
                 temporal_jitter=None):
        self.df = df.reset_index(drop=True)
        self.flow_dir, self.num_frames, self.train = flow_dir, num_frames, train
        self.temporal_jitter = train if temporal_jitter is None else temporal_jitter
        self.camera_comp = camera_comp
        self.transform = ClipTransform(crop=crop, train=train)
        self.labels = self.df.label.astype(int).tolist()
        self.samples = list(zip(self.df.rel_path, self.labels))

    def __len__(self):
        return len(self.df)

    def path(self, i):
        return os.path.join(self.flow_dir, self.df.rel_path[i][:-4] + '.npz')

    def get(self, i, params=None):
        """params (from the RGB dataset) forces the same frames/crop/flip; None samples new ones."""
        z = np.load(self.path(i))
        data, off = z['data'], z['offsets']
        k = (len(off) - 1) // 2
        which = params['frame_idx'] if params else segment_indices(k, self.num_frames, self.temporal_jitter)
        assert max(which) < k, 'RGB and flow caches have different K'
        flow = decode_flow(data, off, which)                       # (T, H, W, 2) in [-1, 1]
        if self.camera_comp:
            flow = flow - np.median(flow, axis=(1, 2), keepdims=True)
        flow, p = self.transform(flow, params)
        if p['flip']:
            flow[..., 0] *= -1                                     # mirrored image: right <-> left
        p['frame_idx'] = which
        return torch.from_numpy(flow).permute(3, 0, 1, 2).float(), self.labels[i], p

    def __getitem__(self, i):
        flow, label, _ = self.get(i)
        return flow, label


class MultimodalClips(Dataset):
    """(rgb (3,T,H,W), flow (2,T,H,W), label) with one shared set of augmentation parameters."""

    def __init__(self, rgb_ds, flow_ds):
        assert list(rgb_ds.df.rel_path) == list(flow_ds.df.rel_path), 'datasets not aligned'
        self.rgb, self.flow = rgb_ds, flow_ds
        self.samples, self.labels = rgb_ds.samples, rgb_ds.labels

    def __len__(self):
        return len(self.rgb)

    def get(self, i):
        rgb, label, params = self.rgb.get(i)
        flow, _, _ = self.flow.get(i, params=params)
        return rgb, flow, label, params

    def __getitem__(self, i):
        rgb, flow, label, _ = self.get(i)
        return rgb, flow, label


### 4.7 Sanity checks: shapes, range, flip correctness, alignment, speed

In [ ]:
import utils.flow_dataset as fds
importlib.reload(fds)

flow_ds = {p: fds.FlowClips(d, LOCAL_FLOW, num_frames=T, train=(p == 'train'), crop=CROP)
           for p, d in phase_df.items()}
f0, l0 = flow_ds['train'][0]
print(f'Flow clip {tuple(f0.shape)} {f0.dtype} | range [{f0.min():.2f}, {f0.max():.2f}] | label {l0}')
assert f0.shape == (2, T, CROP, CROP) and f0.abs().max() <= 1.0 + 1e-6

# Flip correctness: flipped u == -(mirrored u); flipped v == mirrored v
fe = fds.FlowClips(phase_df['train'], LOCAL_FLOW, num_frames=T, train=False, crop=CROP)
base, _, p = fe.get(0)
flipped, _, _ = fe.get(0, params=dict(p, flip=True))
assert torch.equal(flipped[0], -base[0].flip(-1)), 'u not negated on flip'
assert torch.equal(flipped[1], base[1].flip(-1)), 'v changed on flip'
print('✅ Flip handling correct (u negated, v kept)')

# Alignment: multimodal sample shares frames/crop/flip
mm = fds.MultimodalClips(ds['train'], flow_ds['train'])
rgb, flow, label, params = mm.get(0)
assert rgb.shape[1:] == flow.shape[1:], 'RGB and flow differ in T/H/W'
print(f'Multimodal: rgb {tuple(rgb.shape)} + flow {tuple(flow.shape)} | '
      f'crop y={params["y"]}, x={params["x"]}, flip={params["flip"]}')

In [ ]:
# See the alignment: an augmented RGB clip and its flow (same crop/flip/frames)
frames = vds.denormalize(rgb)
flow_px = flow.permute(1, 2, 3, 0).numpy() * FLOW_BOUND
show_t = [0, 5, 10, 15]
mag_max = np.percentile(np.linalg.norm(flow_px, axis=-1), 99) + 1e-6
fig, axes = plt.subplots(2, 4, figsize=(12, 6))
for c, t in enumerate(show_t):
    axes[0, c].imshow(frames[t]); axes[0, c].set_title(f't={t} (cached frame {params["frame_idx"][t]})', fontsize=9)
    axes[1, c].imshow(oflow.flow_to_hsv(flow_px[t], max_mag=mag_max))
for ax in axes.flat:
    ax.axis('off')
plt.suptitle(f'{phase_df["train"].class_name.iloc[0]}: RGB and flow after shared augmentation (flip={params["flip"]})')
plt.tight_layout(); save_fig('flow_rgb_alignment'); plt.show()

print(f'Multimodal loading: {time_loader(mm, min(2, os.cpu_count())):.1f} ms/clip')   # helper from 3.7

**What to check in the figure:** the bright flow regions should sit on the moving parts of the RGB frames directly above them. If they're shifted or mirrored relative to the RGB, the crop or flip isn't being shared.

---
### ✅ Step 4 checklist
- [ ] `utils/optical_flow.py` and `utils/flow_dataset.py` written
- [ ] Flow bound B chosen from data; storage EPE small compared with typical motion
- [ ] Exercise 5 figure (u, v, HSV for 4 consecutive frames)
- [ ] Camera-compensation figure and per-class motion table (evidence for Q2 and Q3)
- [ ] Flow cache built and backed up; `preprocess_config.json` saved
- [ ] Flip and alignment checks pass

**You can now draft Reflection Q1–Q3:** Q1 from Step 2.5 (the sampling trade-off plots and segment sampling as mitigation), Q2 from 4.3, Q3 from 4.4.

**Next — Step 5 (GPU from here on):** the C3D baseline. We'll implement `evaluate()` with top-1/top-5, train with validation-based checkpointing, plot learning curves, and run the label-smoothing sweep.